# Notebook 03: Evaluación y Análisis de Errores


---
### Objetivo
Evaluar el modelo sobre el conjunto de validación propio (40 preguntas distribuidas en 15 fáciles, 15 medias y 10 difíciles) en las bases sistema_academico y clinica_salud.
Calcula la métrica oficial de Exactitud por Ejecución (EX) ponderada por nivel (pesos 1, 2 y 3) y genera el análisis detallado de errores por categoría.

In [1]:
# Setup universal
import os, sys, subprocess
from pathlib import Path

# Detectar si estamos en Google Colab
EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if EN_COLAB:
    import getpass
    REPO = "Albu231311/Deep-Proyecto3"
    NOMBRE = "Albu231311"
    CORREO = "alb231311@uvg.edu.gt"
    if not os.path.exists("/content/repo"):
        token = getpass.getpass("GitHub token: ")
        subprocess.run(f"git clone https://{token}@github.com/{REPO}.git /content/repo", shell=True, check=True)
    os.chdir("/content/repo")
    subprocess.run("git pull", shell=True, check=True)
    subprocess.run(f'git config user.name "{NOMBRE}"', shell=True)
    subprocess.run(f'git config user.email "{CORREO}"', shell=True)
    subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
    ROOT_DIR = Path("/content/repo")
else:
    
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
    ROOT_DIR = Path.cwd().resolve()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f" Entorno listo. Directorio de trabajo: {os.getcwd()} | En Colab: {EN_COLAB}")


 Entorno listo. Directorio de trabajo: /Users/carlitos/Documents/VIII Semestre/Deep Learning/Deep-Proyecto3 | En Colab: False


## 1. Carga de Esquemas y Preguntas de Validación

Cargamos los 2 esquemas en MySQL y leemos datos/validacion/val.jsonl.

In [2]:
import json
from pathlib import Path
from src.mysql_utils import cargar_esquema, ejecutar

esq_dir = Path("datos/validacion/esquemas_val")
if not esq_dir.exists() and Path("../datos/validacion/esquemas_val").exists():
    esq_dir = Path("../datos/validacion/esquemas_val")

cargar_esquema(str(esq_dir / "sistema_academico.sql"), "sistema_academico")
cargar_esquema(str(esq_dir / "clinica_salud.sql"), "clinica_salud")

val_file = Path("datos/validacion/val.jsonl")
if not val_file.exists() and Path("../datos/validacion/val.jsonl").exists():
    val_file = Path("../datos/validacion/val.jsonl")

with open(val_file, "r", encoding="utf-8") as f:
    val_preguntas = [json.loads(line) for line in f]

print(f"Total de preguntas de validación: {len(val_preguntas)}")


Total de preguntas de validación: 40


## 2. Evaluación por Ejecución y Cálculo de EX Ponderado

Evaluamos la referencia con src/evaluador.py para certificar la métrica oficial.

In [3]:
from src.evaluador import evaluar_validacion

def predictor_referencia(esquema, pregunta, ejecutar_fn):
    for p in val_preguntas:
        if p["pregunta"] == pregunta:
            return p["sql_referencia"]
    return "SELECT 1"

db_map = {
    "sistema_academico": "sistema_academico",
    "clinica_salud": "clinica_salud"
}

resultado_eval = evaluar_validacion(val_preguntas, predictor_referencia, db_map)
print(f"EX Ponderado Global : {resultado_eval['ex_ponderado'] * 100:.2f}%")
print(f"EX por Nivel        : {resultado_eval['ex_por_nivel']}")
print(f"Conteo de Errores   : {resultado_eval['conteo_errores']}")


EX Ponderado Global : 100.00%
EX por Nivel        : {'facil': 1.0, 'media': 1.0, 'dificil': 1.0}
Conteo de Errores   : {}


## 3. Exportación de Resultados

Guardamos la tabla detallada en resultados/validacion_propia.csv.

In [4]:
from src.evaluador import guardar_csv

out_csv = Path("resultados/validacion_propia.csv")
if not out_csv.parent.exists() and Path("../resultados").exists():
    out_csv = Path("../resultados/validacion_propia.csv")

guardar_csv(resultado_eval["resultados"], str(out_csv), version="baseline_referencia")
print(f"Resultados guardados en: {out_csv}")


Resultados guardados en: resultados/validacion_propia.csv
